In [7]:
import base64
import os
import pandas as pd
from openai import OpenAI

# Configuration
RUN_SINGLE_TEST_ONLY = False  # Set to False to process all rows
TEST_ROW_INDEX = 1  # Index of the row to test when RUN_SINGLE_TEST_ONLY is True
CSV_FILE_PATH = 'Sample to rate/CSV/AllResults.csv'
OUTPUT_CSV_PATH = 'ResultsWithFaithfulness.csv'
EVALUATION_COLUMN = 'Faithfulness (GPT)'  # Column to check if evaluation exists

# API setup
client = OpenAI()

def encode_image(image_path):
    """Encode an image file to base64 string"""
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except Exception as e:
        print(f"Error encoding image {image_path}: {e}")
        return None

def get_response_with_images(question, image_paths):
    """Get response from OpenAI API with images"""
    print(f"Processing request with {len(image_paths)} images...")
    
    # Filter out None values in case any image encoding failed
    encoded_images = [encode_image(path) for path in image_paths if os.path.exists(path)]
    encoded_images = [img for img in encoded_images if img is not None]
    
    if not encoded_images:
        print("Warning: No valid images to process")
        return "Error: No valid images provided"
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question}
            ] + [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{encoded_image}"}
                } for encoded_image in encoded_images
            ]
        }
    ]
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            max_tokens=1000
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"API Error: {e}")
        return f"Error: {str(e)}"

def load_example_files():
    """Load example files for evaluation context"""
    examples = {}
    
    # Input example
    try:
        with open('Examples/Text/input_text.txt', 'r', encoding='utf-8') as file:
            examples['input'] = file.read()
    except Exception as e:
        print(f"Error loading input example: {e}")
        examples['input'] = "Error loading example"
    
    # Output examples
    for i in range(1, 5):
        try:
            with open(f'Examples/Text/Example{i}.txt', 'r', encoding='utf-8') as file:
                examples[f'output{i}'] = file.read()
        except Exception as e:
            print(f"Error loading output example {i}: {e}")
            examples[f'output{i}'] = f"Error loading example {i}"
    
    return examples

def should_skip_row(row, column_name):
    """
    Determine if a row should be skipped based on whether it already has an evaluation
    
    Args:
        row: DataFrame row
        column_name: Name of the column to check
        
    Returns:
        bool: True if the row should be skipped, False otherwise
    """
    # Check if column exists in the row
    if column_name not in row:
        return False
    
    # Get the value and handle different types properly
    value = row[column_name]
    
    # Check if the value is empty or NaN
    if pd.isna(value):
        return False
    
    # If it's a number (int, float) and not zero, consider it filled
    if isinstance(value, (int, float)):
        return value > 0
    
    # If it's a string, check if it's not just whitespace
    if isinstance(value, str) and value.strip():
        return True
    
    # Default: don't skip
    return False

def main():
    print(f"Starting faithfulness evaluation process...")
    print(f"{'SINGLE TEST MODE' if RUN_SINGLE_TEST_ONLY else 'BULK PROCESSING MODE'}")
    
    # Load CSV data
    try:
        df = pd.read_csv(CSV_FILE_PATH, sep=',')
        print(f"Loaded CSV with {len(df)} rows")
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return
    
    # Load example files
    examples = load_example_files()
    explanations = "followed by your step-by-step reasoning"
    example_image_path = os.path.join('Examples/Images', 'Picture Example.png')
    
    # Ensure the evaluation column exists
    if EVALUATION_COLUMN not in df.columns:
        df[EVALUATION_COLUMN] = ""
        print(f"Created missing column: {EVALUATION_COLUMN}")
    
    # Prepare rows to process
    if RUN_SINGLE_TEST_ONLY:
        if TEST_ROW_INDEX >= len(df):
            print(f"Error: Test row index {TEST_ROW_INDEX} is out of bounds (max index: {len(df)-1})")
            return
        rows_to_process = df.iloc[TEST_ROW_INDEX:TEST_ROW_INDEX+1].iterrows()
        print(f"Running single test for row index {TEST_ROW_INDEX}")
    else:
        rows_to_process = df.iterrows()
        print("Processing all rows...")
    
    # Process rows
    processed_count = 0
    skipped_count = 0
    
    for index, row in rows_to_process:
        # Check if this row should be skipped
        if should_skip_row(row, EVALUATION_COLUMN):
            print(f"Row {index}: Skipping - already has value '{row[EVALUATION_COLUMN]}' in {EVALUATION_COLUMN}")
            skipped_count += 1
            continue
        
        print(f"\nProcessing row {index}...")
        
        # Extract values from the row
        input_text = row.get('Input', '')
        report = row.get('Output', '')
        case_study = row.get('Case study', '')
        
        # Build the prompt
        question = f"""Your task is to rate a report based on its faithfulness with respect to an input context and input plots. Faithfulness is on a scale from 1 (worst) to 5 (perfect). Your answer must state the number you give for faithfulness {explanations}.
        Consider the following examples. For example, the input context is: {examples['input']} and the input plot is given to you as an attachment and tagged with the label "Image used for the example".
        An example of a report is: {examples['output1']}. This example scores 3 on faithfulness, because it states several values incorrectly and it does not mention the final values shown in the plot. 
        A second example of a report is: {examples['output2']}. This second example scores 3 on faithfulness, because it states several values incorrectly and it does not mention the final values shown in the plot. 
        A third example of a report is: {examples['output3']}. This third example scores 1 on faithfulness, because there it lacks context, it omits the overall evolution, it states several values incorrectly, and it misses intermediate trends from the plot. 
        A last example is: {examples['output4']}. This last report scores 1 on faithfulness, because there it lacks context, it omits the overall evolution, it states several values incorrectly, and it misses intermediate trends from the plot.
        In your case, the input context is: {input_text} The accompanying input plots are given to you as an attachment. 
        The report that you must rate for faithfulness is {report}"""
        
        # Get image paths
        image_paths = []
        case_study_image_dir = f'Sample to rate/Images/{case_study}'
        
        if os.path.exists(case_study_image_dir) and os.path.isdir(case_study_image_dir):
            image_paths = [os.path.join(case_study_image_dir, f) for f in os.listdir(case_study_image_dir) 
                           if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            print(f"Found {len(image_paths)} images in case study directory")
        else:
            print(f"Case study image directory not found: {case_study_image_dir}")
        
        # Add example image
        if os.path.exists(example_image_path):
            image_paths.append(example_image_path)
            print(f"Added example image: {example_image_path}")
        else:
            print(f"Example image not found: {example_image_path}")
        
        if not image_paths:
            print("No images found for this row. Skipping...")
            continue
        
        # Get response from API
        api_response = get_response_with_images(question, image_paths)
        
        # Update dataframe with results
        df.at[index, 'LLM Faithfulness Raw Response'] = api_response
        df.at[index, EVALUATION_COLUMN] = extract_faithfulness_score(api_response)
        df.at[index, 'Explanations for Evaluation'] = 'Yes'
        df.at[index, 'Number of examples for evaluation'] = 4
        
        processed_count += 1
        print(f"Completed row {index}")
        
        # Save after each row to preserve progress
        df.to_csv(OUTPUT_CSV_PATH, index=False)
        print(f"Saved progress to {OUTPUT_CSV_PATH}")
        
        # Break after first row if in test mode
        if RUN_SINGLE_TEST_ONLY:
            break
    
    # Final report
    print(f"\nEvaluation complete:")
    print(f"  - Total rows processed: {processed_count}")
    print(f"  - Rows skipped (already evaluated): {skipped_count}")
    print(f"  - Results saved to: {OUTPUT_CSV_PATH}")

def extract_faithfulness_score(response_text):
    """Extract the faithfulness score (1-5) from the API response"""
    import re
    
    # Common patterns for score reporting
    patterns = [
        r"faithfulness(?:\s+score)?(?:\s+of)?(?:\s+is)?(?:\s*:)?\s*(\d)(?:\.0*)?(?:/5)?",
        r"(?:score|rating|give|assign)(?:\s+a)?(?:\s+faithfulness)?(?:\s+score)?(?:\s+of)?\s*:?\s*(\d)(?:\.0*)?(?:/5)?",
        r"(\d)(?:\.0*)?(?:/5)?(?:\s+out\s+of\s+5)?(?:\s+for\s+faithfulness)"
    ]
    
    # Try each pattern
    for pattern in patterns:
        matches = re.search(pattern, response_text, re.IGNORECASE)
        if matches:
            return matches.group(1)
    
    # If no match found, look for any digit followed by "out of 5"
    matches = re.search(r"(\d)(?:\s+out\s+of\s+5)", response_text)
    if matches:
        return matches.group(1)
    
    # Fall back to just looking for a single digit that might be the score
    matches = re.findall(r'\b(\d)\b', response_text)
    for match in matches:
        if 1 <= int(match) <= 5:
            return match
    
    # If no score found, return empty string
    return ""

if __name__ == "__main__":
    main()

Starting faithfulness evaluation process...
BULK PROCESSING MODE
Loaded CSV with 144 rows
Processing all rows...
Row 0: Skipping - already has value '2.0' in Faithfulness (GPT)
Row 1: Skipping - already has value '5.0' in Faithfulness (GPT)
Row 2: Skipping - already has value '3.0' in Faithfulness (GPT)
Row 3: Skipping - already has value '3.0' in Faithfulness (GPT)
Row 4: Skipping - already has value '3.0' in Faithfulness (GPT)
Row 5: Skipping - already has value '4.0' in Faithfulness (GPT)
Row 6: Skipping - already has value '2.0' in Faithfulness (GPT)
Row 7: Skipping - already has value '3.0' in Faithfulness (GPT)
Row 8: Skipping - already has value '4.0' in Faithfulness (GPT)
Row 9: Skipping - already has value '4.0' in Faithfulness (GPT)
Row 10: Skipping - already has value '3.0' in Faithfulness (GPT)
Row 11: Skipping - already has value '3.0' in Faithfulness (GPT)
Row 12: Skipping - already has value '5.0' in Faithfulness (GPT)
Row 13: Skipping - already has value '3.0' in Faithf